# 🚀 POLYDIM V4.2 - AGENTE EMISOR (SENDER)
Implementa PMTP con Isometría Unitaria (FFT + Fase), eliminando FWHT para evitar propagación de error.


In [ ]:
!pip install -q torch numpy
import torch
import numpy as np
import time
import os
import random
from google.colab import drive
import math

drive.mount('/content/drive')
BUS_DIR = "/content/drive/MyDrive/POLYDIM_BUS"
os.makedirs(BUS_DIR, exist_ok=True)
print(f"✅ Bus configurado en: {BUS_DIR}")



In [ ]:
# MOTOR ISOMÉTRICO V4.2 (FFT PURA)
class IsometricRotation(torch.nn.Module):
    def __init__(self, D: int, seed=42):
        super().__init__()
        self.D = D
        # Phase scale
        phase_scale = min(0.01, 1.0 / math.sqrt(D))
        gen = torch.Generator().manual_seed(seed)
        self.phase = torch.nn.Parameter(torch.randn(D, generator=gen) * phase_scale)
        self.phase.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.fft.fft(x, dim=-1)
        x = x * torch.exp(1j * self.phase).unsqueeze(0)
        x = torch.fft.ifft(x, dim=-1)
        return x.real



In [ ]:
# TRANSFERENCIA ESTOCÁSTICA V4.2
print("🚀 INICIANDO TRANSFERENCIA DE ECPs (FFT Pura)...")
dim_sizes = [64, 256, 1024, 4096]

for chunk_id in range(1, 101):
    d = random.choice(dim_sizes)
    seed_rot = 42 + chunk_id

    ecp_base = torch.randn(1, d)
    ecp_base = ecp_base / torch.norm(ecp_base, dim=-1, keepdim=True)

    t0 = time.time()
    rotor = IsometricRotation(d, seed=seed_rot)
    ecp_encoded = rotor.forward(ecp_base)
    t_encode = (time.time() - t0) * 1000

    file_path = os.path.join(BUS_DIR, f"chunk_{chunk_id:04d}.pt")
    torch.save({
        'tensor': ecp_encoded,
        'seed_rot': seed_rot,
        'original_for_validation_only': ecp_base
    }, file_path)

    if chunk_id % 10 == 0:
        print(f"📤 Enviado Chunk {chunk_id:04d} | D={d:<4} | Encode={t_encode:.2f}ms")
print("✅ SENDER FINALIZADO.")

